# 05 — Decision Tree Models

Project: E-Waste Profitability Prediction
Group: 2026-AI-14
Module: IT3091 — Machine Learning

Regression: DecisionTreeRegressor
Classification: DecisionTreeClassifier

Decision trees can capture nonlinear relationships and feature
interactions. Tree depth and minimum leaf size are tuned to
control overfitting.

Validation:
- Preserve the existing train/test split.
- Use the same five-fold settings as Notebook 04.
- Fit preprocessing inside each CV training fold.
- Compare raw-only and engineered feature sets.
- Keep the test dataset untouched during model selection.

Batch_Priority represents derived profit tiers.

Libraries import

In [ ]:
import pandas as pd
import numpy as np
import json
import joblib
import platform
import sklearn

from google.colab import files

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.tree import (
    DecisionTreeRegressor,
    DecisionTreeClassifier
)

from sklearn.model_selection import (
    KFold,
    StratifiedKFold,
    GridSearchCV
)

pd.set_option("display.max_columns", None)

print("Python:", platform.python_version())
print("scikit-learn:", sklearn.__version__)

Python: 3.13.15
scikit-learn: 1.6.1


In [ ]:
uploaded = files.upload()

Saving train_features.csv to train_features.csv
Saving test_features.csv to test_features.csv
Saving feature_config.json to feature_config.json


In [ ]:
required_files = [
    "train_features.csv",
    "test_features.csv",
    "feature_config.json"
]

for filename in required_files:
    assert filename in uploaded, f"Missing file: {filename}"

train_df = pd.read_csv("train_features.csv")
test_df = pd.read_csv("test_features.csv")

with open("feature_config.json") as file:
    config = json.load(file)

print("Training dataset:", train_df.shape)
print("Test dataset:", test_df.shape)

Training dataset: (16000, 19)
Test dataset: (4000, 19)


Inputs and labels validation

In [ ]:
for dataset in [train_df, test_df]:
    assert dataset["device_id"].notna().all()
    assert dataset["device_id"].is_unique

assert set(train_df["device_id"]).isdisjoint(
    set(test_df["device_id"])
)

assert len(train_df) == 16000
assert len(test_df) == 4000

all_features = config["feature_columns"]
raw_features = config["raw_feature_columns"]

forbidden_inputs = {
    "device_id",
    "item_name",
    "total_profit_usd",
    "Batch_Priority"
}

assert not set(all_features) & forbidden_inputs

X = train_df[all_features].copy()
y_reg = train_df["total_profit_usd"].copy()
y_cls = train_df["Batch_Priority"].copy()

assert np.isfinite(y_reg).all()
assert y_cls.notna().all()
assert set(y_cls.unique()) == {"Low", "Medium", "High"}

policy = config["priority_policy"]

expected_labels = np.where(
    y_reg <= policy["low_cutoff_usd"],
    "Low",
    np.where(
        y_reg <= policy["high_cutoff_usd"],
        "Medium",
        "High"
    )
)

assert np.array_equal(
    y_cls.to_numpy(),
    expected_labels
)

print("Consistency checks passed.")
print("Training inputs:", X.shape)

Consistency checks passed.
Training inputs: (16000, 15)


Tree preprocessing pipeline

In [ ]:
def make_tree_pipeline(model, selected_features):
    numeric = [
        column
        for column in config["numeric_features"]
        if column in selected_features
    ]

    categorical = [
        column
        for column in config["categorical_features"]
        if column in selected_features
    ]

    numeric_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ])

    categorical_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ])

    preprocessor = ColumnTransformer([
        ("numeric", numeric_pipeline, numeric),
        ("categorical", categorical_pipeline, categorical)
    ])

    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

Create CV folds

In [ ]:
regression_cv = list(
    KFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    ).split(X, y_reg)
)

classification_cv = list(
    StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    ).split(X, y_cls)
)

feature_sets = {
    "raw_only": raw_features,
    "with_engineered": all_features
}

print("Five-fold CV configured.")

Five-fold CV configured.


Decision Tree Regression tune

In [ ]:
regression_grid = {
    "model__max_depth": [6, 12, None],
    "model__min_samples_leaf": [2, 10, 30]
}

regression_searches = {}
regression_rows = []

for name, selected_features in feature_sets.items():
    pipeline = make_tree_pipeline(
        DecisionTreeRegressor(random_state=42),
        selected_features
    )

    search = GridSearchCV(
        estimator=pipeline,
        param_grid=regression_grid,
        scoring={
            "mae": "neg_mean_absolute_error",
            "rmse": "neg_root_mean_squared_error",
            "r2": "r2"
        },
        refit="mae",
        cv=regression_cv,
        n_jobs=2,
        return_train_score=True,
        error_score="raise"
    )

    search.fit(X, y_reg)
    regression_searches[name] = search

    results = search.cv_results_
    index = search.best_index_

    regression_rows.append({
        "feature_set": name,
        "CV_MAE_USD": -results["mean_test_mae"][index],
        "CV_MAE_STD": results["std_test_mae"][index],
        "CV_RMSE_USD": -results["mean_test_rmse"][index],
        "CV_R2": results["mean_test_r2"][index],
        "Train_MAE_USD": -results["mean_train_mae"][index],
        "best_parameters": json.dumps(search.best_params_)
    })

    pd.DataFrame(results).to_csv(
        f"05_regression_candidates_{name}.csv",
        index=False
    )

regression_comparison = pd.DataFrame(
    regression_rows
).sort_values("CV_MAE_USD")

display(regression_comparison)

best_reg_feature_set = (
    regression_comparison.iloc[0]["feature_set"]
)

best_reg_search = regression_searches[
    best_reg_feature_set
]

best_reg_model = best_reg_search.best_estimator_

print("Selected regression features:", best_reg_feature_set)
print("Selected parameters:", best_reg_search.best_params_)

,feature_set,CV_MAE_USD,CV_MAE_STD,CV_RMSE_USD,CV_R2,Train_MAE_USD,best_parameters
0,raw_only,0.339261,0.007714,0.761429,0.996539,0.254431,"{""model__max_depth"": null, ""model__min_samples..."
1,with_engineered,0.341849,0.007132,0.764515,0.996512,0.252396,"{""model__max_depth"": null, ""model__min_samples..."


Selected regression features: raw_only
Selected parameters: {'model__max_depth': None, 'model__min_samples_leaf': 10}


Tune Decision Treen Classification

In [ ]:
classification_grid = {
    "model__max_depth": [6, 12, None],
    "model__min_samples_leaf": [2, 10, 30],
    "model__class_weight": [None, "balanced"]
}

classification_searches = {}
classification_rows = []

for name, selected_features in feature_sets.items():
    pipeline = make_tree_pipeline(
        DecisionTreeClassifier(random_state=42),
        selected_features
    )

    search = GridSearchCV(
        estimator=pipeline,
        param_grid=classification_grid,
        scoring={
            "macro_f1": "f1_macro",
            "balanced_accuracy": "balanced_accuracy",
            "accuracy": "accuracy"
        },
        refit="macro_f1",
        cv=classification_cv,
        n_jobs=2,
        return_train_score=True,
        error_score="raise"
    )

    search.fit(X, y_cls)
    classification_searches[name] = search

    results = search.cv_results_
    index = search.best_index_

    classification_rows.append({
        "feature_set": name,
        "CV_Macro_F1": results["mean_test_macro_f1"][index],
        "CV_Macro_F1_STD": results["std_test_macro_f1"][index],
        "CV_Balanced_Accuracy": (
            results["mean_test_balanced_accuracy"][index]
        ),
        "CV_Accuracy": results["mean_test_accuracy"][index],
        "Train_Macro_F1": results["mean_train_macro_f1"][index],
        "best_parameters": json.dumps(search.best_params_)
    })

    pd.DataFrame(results).to_csv(
        f"05_classification_candidates_{name}.csv",
        index=False
    )

classification_comparison = pd.DataFrame(
    classification_rows
).sort_values("CV_Macro_F1", ascending=False)

display(classification_comparison)

best_cls_feature_set = (
    classification_comparison.iloc[0]["feature_set"]
)

best_cls_search = classification_searches[
    best_cls_feature_set
]

best_cls_model = best_cls_search.best_estimator_

print("Selected classification features:", best_cls_feature_set)
print("Selected parameters:", best_cls_search.best_params_)

,feature_set,CV_Macro_F1,CV_Macro_F1_STD,CV_Balanced_Accuracy,CV_Accuracy,Train_Macro_F1,best_parameters
1,with_engineered,0.978937,0.001988,0.978936,0.978938,0.987460,"{""model__class_weight"": ""balanced"", ""model__ma..."
0,raw_only,0.978321,0.001597,0.978311,0.978313,0.986978,"{""model__class_weight"": ""balanced"", ""model__ma..."


Selected classification features: with_engineered
Selected parameters: {'model__class_weight': 'balanced', 'model__max_depth': 12, 'model__min_samples_leaf': 10}


Fold Results

In [ ]:
reg_results = best_reg_search.cv_results_
reg_index = best_reg_search.best_index_

regression_fold_results = pd.DataFrame({
    "Fold": range(1, 6),
    "MAE_USD": [
        -reg_results[f"split{i}_test_mae"][reg_index]
        for i in range(5)
    ],
    "RMSE_USD": [
        -reg_results[f"split{i}_test_rmse"][reg_index]
        for i in range(5)
    ],
    "R2": [
        reg_results[f"split{i}_test_r2"][reg_index]
        for i in range(5)
    ]
})

cls_results = best_cls_search.cv_results_
cls_index = best_cls_search.best_index_

classification_fold_results = pd.DataFrame({
    "Fold": range(1, 6),
    "Macro_F1": [
        cls_results[f"split{i}_test_macro_f1"][cls_index]
        for i in range(5)
    ],
    "Balanced_Accuracy": [
        cls_results[f"split{i}_test_balanced_accuracy"][cls_index]
        for i in range(5)
    ],
    "Accuracy": [
        cls_results[f"split{i}_test_accuracy"][cls_index]
        for i in range(5)
    ]
})

display(regression_fold_results)
display(classification_fold_results)

,Fold,MAE_USD,RMSE_USD,R2
0,1,0.338948,0.776009,0.996541
1,2,0.330263,0.758952,0.996526
2,3,0.350592,0.835463,0.995804
3,4,0.331692,0.691308,0.997036
4,5,0.344809,0.745414,0.996791


,Fold,Macro_F1,Balanced_Accuracy,Accuracy
0,1,0.979683,0.979691,0.979688
1,2,0.981559,0.981562,0.981563
2,3,0.975921,0.975933,0.975938
3,4,0.977502,0.977494,0.977500
4,5,0.980022,0.979998,0.980000


overfitting indicators

In [ ]:
reg_train_mae = -reg_results[
    "mean_train_mae"
][reg_index]

reg_cv_mae = -reg_results[
    "mean_test_mae"
][reg_index]

cls_train_f1 = cls_results[
    "mean_train_macro_f1"
][cls_index]

cls_cv_f1 = cls_results[
    "mean_test_macro_f1"
][cls_index]

overfitting_summary = pd.DataFrame([
    {
        "task": "Regression",
        "metric": "MAE USD",
        "training_score": reg_train_mae,
        "cv_score": reg_cv_mae,
        "gap": reg_cv_mae - reg_train_mae
    },
    {
        "task": "Classification",
        "metric": "Macro-F1",
        "training_score": cls_train_f1,
        "cv_score": cls_cv_f1,
        "gap": cls_train_f1 - cls_cv_f1
    }
])

display(overfitting_summary)

print(
    "Selected regression tree depth:",
    best_reg_model.named_steps["model"].get_depth()
)

print(
    "Selected classification tree depth:",
    best_cls_model.named_steps["model"].get_depth()
)

,task,metric,training_score,cv_score,gap
0,Regression,MAE USD,0.254431,0.339261,0.084830
1,Classification,Macro-F1,0.987460,0.978937,0.008523


Selected regression tree depth: 22
Selected classification tree depth: 12


Decision Tree Regression achieved a mean CV MAE of USD 0.339261, approximately 82.9% lower than the Linear Regression baseline. Its training MAE was USD 0.254431, giving a train–CV gap of USD 0.084830. Decision Tree Classification achieved a CV macro-F1 of 0.978937, slightly above the Logistic Regression baseline, with a train–CV gap of 0.008523. The fitted regression and classification trees had depths of 22 and 12 respectively. These results support retaining both trees as candidates; final model selection and held-out test evaluation remain pending.

Baseline comparison (summery)

In [ ]:
model_summary = {
    "notebook": "05_Decision_Tree_Models",
    "regression_model": "DecisionTreeRegressor",
    "classification_model": "DecisionTreeClassifier",

    "regression_feature_set": best_reg_feature_set,
    "classification_feature_set": best_cls_feature_set,

    "regression_features": feature_sets[best_reg_feature_set],
    "classification_features": feature_sets[best_cls_feature_set],

    "regression_cv_mae_usd": float(
        -best_reg_search.best_score_
    ),
    "regression_cv_rmse_usd": float(
        -reg_results["mean_test_rmse"][reg_index]
    ),
    "regression_cv_r2": float(
        reg_results["mean_test_r2"][reg_index]
    ),

    "classification_cv_macro_f1": float(
        best_cls_search.best_score_
    ),
    "classification_cv_balanced_accuracy": float(
        cls_results["mean_test_balanced_accuracy"][cls_index]
    ),
    "classification_cv_accuracy": float(
        cls_results["mean_test_accuracy"][cls_index]
    ),

    "regression_parameters": best_reg_search.best_params_,
    "classification_parameters": best_cls_search.best_params_,

    "test_evaluated": False,
    "priority_meaning": "Profit-tier proxy",
    "sklearn_version": sklearn.__version__
}

display(pd.DataFrame([model_summary]))

,notebook,regression_model,classification_model,regression_feature_set,classification_feature_set,regression_features,classification_features,regression_cv_mae_usd,regression_cv_rmse_usd,regression_cv_r2,classification_cv_macro_f1,classification_cv_balanced_accuracy,classification_cv_accuracy,regression_parameters,classification_parameters,test_evaluated,priority_meaning,sklearn_version
0,05_Decision_Tree_Models,DecisionTreeRegressor,DecisionTreeClassifier,raw_only,with_engineered,"[weight_g, device_age_years, gold_yield_g, sil...","[weight_g, device_age_years, gold_yield_g, sil...",0.339261,0.761429,0.996539,0.978937,0.978936,0.978938,"{'model__max_depth': None, 'model__min_samples...","{'model__class_weight': 'balanced', 'model__ma...",False,Profit-tier proxy,1.6.1


Models and evidace save

In [ ]:
regression_comparison.to_csv(
    "05_regression_feature_comparison.csv",
    index=False
)

classification_comparison.to_csv(
    "05_classification_feature_comparison.csv",
    index=False
)

regression_fold_results.to_csv(
    "05_regression_cv_folds.csv",
    index=False
)

classification_fold_results.to_csv(
    "05_classification_cv_folds.csv",
    index=False
)

overfitting_summary.to_csv(
    "05_overfitting_summary.csv",
    index=False
)

with open("05_model_summary.json", "w") as file:
    json.dump(model_summary, file, indent=2)

joblib.dump(
    best_reg_model,
    "05_tree_profit_pipeline.joblib"
)

joblib.dump(
    best_cls_model,
    "05_tree_priority_pipeline.joblib"
)

print("Decision Tree models and CV evidence saved.")
print("Test evaluation has not been performed.")

Decision Tree models and CV evidence saved.
Test evaluation has not been performed.


## Decision Tree interpretation

### Regression
- Selected feature set:
- Selected depth and minimum leaf size:
- Mean CV MAE, RMSE and R²:
- Comparison with Linear Regression:
- Training/CV performance gap:

### Classification
- Selected feature set:
- Selected depth, minimum leaf size and class weighting:
- Mean CV macro-F1 and balanced accuracy:
- Comparison with Logistic Regression:
- Training/CV performance gap:

### Limitations
- CV results were used for tuning and feature selection.
- Final test evaluation is pending.
- A single tree can be sensitive to changes in training data.
- Priority labels are derived profit tiers.
- Classifier CV is conditional on the training-derived tier policy.

### AI-use declaration
Record the assistance actually used and the student's own
verification, interpretation and contribution.

In [ ]:
files.download("05_model_summary.json")
files.download("05_regression_cv_folds.csv")
files.download("05_classification_cv_folds.csv")
files.download("05_overfitting_summary.csv")

files.download("05_tree_profit_pipeline.joblib")
files.download("05_tree_priority_pipeline.joblib")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
train_df.to_csv("05_train_features.csv", index=False)
test_df.to_csv("05_test_features.csv", index=False)

files.download("05_train_features.csv")
files.download("05_test_features.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>